# Machine Learning Cycle - Image Classification
This notebook documents data acquisition, preprocessing, model training, optimization, testing metrics, prediction, and retraining trigger flow.

In [7]:
from pathlib import Path
import json
import sys
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
sys.path.append(str(project_root / 'src'))
print('Project root:', project_root)

Project root: /Users/michaelkimani/Desktop/Kaggle Folder


## 1. Data Acquisition
Dataset is image-based (non-tabular) and stored in train/test class folders under archive (14).

In [8]:
from preprocessing import resolve_default_paths

paths = resolve_default_paths(project_root)
print('Train dir:', paths.train_dir)
print('Test dir :', paths.test_dir)

Train dir: /Users/michaelkimani/Desktop/Kaggle Folder/archive (14)/seg_train/seg_train
Test dir : /Users/michaelkimani/Desktop/Kaggle Folder/archive (14)/seg_test/seg_test


## 2. Data Processing
Preprocessing steps:
- resize images to fixed size
- convert RGB pixels into normalized vectors
- derive interpretable features: brightness, blue_ratio, green_ratio, texture_strength

In [9]:
from preprocessing import load_split, sample_story_features

x_train, y_train, classes = load_split(paths.train_dir, max_per_class=200)
x_test, y_test, _ = load_split(paths.test_dir, max_per_class=100)
story_x, story_y = sample_story_features(paths.test_dir, classes, samples_per_class=20)

print('Train shape:', x_train.shape)
print('Test shape :', x_test.shape)
print('Classes    :', classes)
print('Feature rows:', story_x.shape[0])

Train shape: (1200, 3072)
Test shape : (600, 3072)
Classes    : ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
Feature rows: 120


## 3. Model Creation and Optimization
Optimization choices used:
- `LogisticRegression` with regularization (default L2)
- `solver='saga'` to handle multiclass efficiently
- tuned `max_iter` for better convergence
- standardization before classification

In [10]:
model_path = project_root / 'models' / 'scene_classifier.keras'
meta_path = project_root / 'models' / 'scene_classifier_meta.json'
metrics_path = project_root / 'results' / 'metrics.json'

print('Model file exists:', model_path.exists())
print('Meta file exists :', meta_path.exists())
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
print('Model type:', metrics.get('model_type', 'unknown'))
print('Training config:', metrics.get('config', {}))

Model file exists: True
Meta file exists : True
Model type: efficientnetb0_transfer_learning
Training config: {'image_size': [224, 224], 'batch_size': 16, 'head_epochs': 1, 'fine_tune_epochs': 1, 'unfreeze_last_layers': 20, 'learning_rate': 0.001, 'fine_tune_learning_rate': 1e-05, 'dropout_rate': 0.4, 'dense_units': 256, 'validation_split': 0.1, 'random_state': 42}


## 4. Model Testing and Evaluation Metrics
Explicit metrics reported:
- Accuracy
- Precision
- Recall
- F1-score
Additional metric:
- Confusion Matrix

In [11]:
metrics_path = project_root / 'results' / 'metrics.json'
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))

print('Accuracy:', metrics['accuracy'])
report_df = pd.DataFrame(metrics['classification_report']).transpose()
display(report_df[['precision', 'recall', 'f1-score']].head(8))
print('Confusion Matrix size:', len(metrics['confusion_matrix']), 'x', len(metrics['confusion_matrix'][0]))

Accuracy: 0.9123333333333333


,precision,recall,f1-score
buildings,0.970667,0.832952,0.896552
forest,1.000000,0.993671,0.996825
glacier,0.860037,0.844485,0.852190
mountain,0.874510,0.849524,0.861836
sea,0.934457,0.978431,0.955939
street,0.864198,0.978044,0.917603
accuracy,0.912333,0.912333,0.912333
macro avg,0.917311,0.912851,0.913491


Confusion Matrix size: 6 x 6


### Interpretation of Results
Using transfer learning significantly improved performance over the earlier baseline.
Current results are above 90% accuracy, with strong weighted precision, recall, and F1.
Remaining confusion is mostly between visually related scene classes (for example glacier and mountain).

## 5. Prediction Function Demo

In [17]:
import json
import subprocess

sample_image = paths.test_dir / 'forest' / '20056.jpg'
cmd = [
    str(project_root / '.venv312' / 'bin' / 'python'),
    str(project_root / 'scripts' / 'predict_keras.py'),
    str(project_root),
    str(sample_image),
]
result = subprocess.run(cmd, capture_output=True, text=True, check=False)
prediction_output = json.loads(result.stdout)[0] if result.stdout else {'error': result.stderr}
print('Sample image:', sample_image.name)
prediction_output

Sample image: 20056.jpg


{'image': '20056.jpg',
 'predicted_class': 'forest',
 'confidence': 0.9990248680114746,
 'top_3': [{'class': 'forest', 'probability': 0.9990248680114746},
  {'class': 'street', 'probability': 0.0006823482690379024},
  {'class': 'glacier', 'probability': 0.00021934081451036036}],
 'model_type': 'EfficientNetB0'}

## 6. Retraining Trigger
Retraining flow for grading:
1. Upload images through dashboard/API (`/upload-bulk`) and save into `data/uploads/`.
2. Preprocessing is applied through `src/preprocessing.py`.
3. Retrain via `POST /retrain` or `python src/retrain.py`, reusing the established pipeline and regenerating model + metrics artifacts.